## Chapter 3 Exercises

Use the following workbook to practice what you've learned. There are 2 exercises here. Please feel free to extend the existing scenarios, experiment and test other samples you find while reading [the Agent Framework documentation](https://learn.microsoft.com/en-us/agent-framework/).

### Client Setup

Make sure you have `client` available from the Section 1 notebook. If not, run the cell below.

In [ ]:
import os
from dotenv import load_dotenv
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

load_dotenv(override=True)

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)
print("Client ready.")

### Exercise 1 - Create a Multi-Agent Workflow

Build your own workflow with **3 agents** that collaborate on a task. Some scenario ideas:

| Scenario | Agent 1 | Agent 2 | Agent 3 |
|----------|---------|---------|----------|
| Business case validation | Analyst (drafts case) | Finance (reviews risks) | Manager (approves/rejects) |
| Content pipeline | Writer (creates draft) | Editor (improves clarity) | Fact-checker (verifies claims) |
| Code review | Developer (writes code) | Reviewer (suggests improvements) | Security (checks vulnerabilities) |

**Hints:**
- Create each agent with `Agent(client=client, name=..., instructions=...)`
- Pass agents directly to `WorkflowBuilder` -- the builder auto-wraps them in `AgentExecutor`
- Use `WorkflowBuilder(start_executor=agent1)` and `.add_edge(agent1, agent2)` to connect them
- The workflow is linear: Agent1 -> Agent2 -> Agent3

In [ ]:
from agent_framework import Agent, WorkflowBuilder, WorkflowEvent

# Step 1: Create your agents
agent1 = Agent(
    client=client,
    name="agent1",
    instructions="TODO: Add instructions for your first agent",
)

agent2 = Agent(
    client=client,
    name="agent2",
    instructions="TODO: Add instructions for your second agent",
)

agent3 = Agent(
    client=client,
    name="agent3",
    instructions="TODO: Add instructions for your third agent",
)

# Step 2: Build the workflow
workflow = (
    WorkflowBuilder(start_executor=agent1)
    # TODO: Connect your agents
    .build()
)

 # TODO: Run the workflow and print agent outputs:
result = 

print(f"Workflow State: {result.get_final_state()}")

<details>
<summary>See the solution</summary>

```python
from agent_framework import Agent, WorkflowBuilder

analyst = Agent(
    client=client,
    name="analyst",
    instructions="You are a business analyst. Draft a short business case based on the provided idea.",
)

finance = Agent(
    client=client,
    name="finance",
    instructions="You are a finance expert. Review the business case and highlight any financial risks or constraints.",
)

approval = Agent(
    client=client,
    name="approval",
    instructions=(
        "You are a senior manager. Decide whether to approve the business case based on strategic alignment. "
        "Respond with 'Approved' or 'Needs Revision' and provide reasoning."
    ),
)

workflow = (
    WorkflowBuilder(start_executor=analyst)
    .add_edge(analyst, finance)
    .add_edge(finance, approval)
    .build()
)

result = await workflow.run("Develop a business case for implementing an AI-driven customer support service.")

for event in result:
    if event.type == "executor_completed":
        print(f"\nExecutor: {event.executor_id}")
        print(f"Inspect:\n{event.data}")
        print("-" * 40)
    if event.type == "output":
        print(f"\nWorkflow output: {event.data}")

print(f"Workflow State: {result.get_final_state()}")
```
</details>

### Exercise 2: Build a Custom Executor Workflow with Streaming

Create a workflow using **custom executors** that process a support ticket. The workflow should:

1. **Classifier** - Analyzes the ticket and categorizes it (billing, technical, general)
2. **Responder** - Generates an appropriate response based on the classification

Use `workflow.run(..., stream=True)` to observe the workflow events in real-time.

**Hints:**
- Create a custom `Classifier` executor that receives a string and sends a `dict` with the ticket and category
- Create a custom `Responder` executor that receives the `dict` and yields the final response
- Pass executor instances directly to `WorkflowBuilder(start_executor=...)` and `.add_edge()`
- Events are `WorkflowEvent` objects -- check `event.event_type` (e.g. `"status"`, `"output"`)

In [ ]:
from agent_framework import Agent, Executor, WorkflowBuilder, WorkflowContext, handler
from typing_extensions import Never

# Step 1: Define custom Classifier executor
class Classifier(Executor):
    def __init__(self, chat_client, id: str = "classifier"):
        self.agent = Agent(
            client=chat_client,
            name="classifier",
            instructions="TODO: Add classifier instructions",
        )
        super().__init__(id=id)

    @handler
    async def handle(self, ticket: str, ctx: WorkflowContext[dict]) -> None:
        # TODO: Run the agent
        # TODO: Extract the category
        # TODO: Send data via ctx.send_message()
        pass


# Step 2: Define custom Responder executor
class Responder(Executor):
    def __init__(self, chat_client, id: str = "responder"):
        self.agent = Agent(
            client=chat_client,
            name="responder",
            instructions="TODO: Add responder instructions",
        )
        super().__init__(id=id)

    @handler
    async def handle(self, data: dict, ctx: WorkflowContext[Never, str]) -> None:
        # TODO: Build a prompt 
        # TODO: Run agent
        # TODO: Get the final response
        pass


# Step 3: Create instances
classifier = Classifier(client)
responder = Responder(client)

# Step 4: Build workflow
workflow = (
    WorkflowBuilder(start_executor=classifier)
    # TODO: Connect classifier to responder using .add_edge()
    .build()
)

# Step 5: Run with streaming and handle events
async for event in workflow.run(
    "I was charged twice for my subscription last month. Please help!",
    stream=True,
):
    # TODO: Use event.type to inspect events and get output:



<details>
<summary>See the solution</summary>

```python
from agent_framework import Agent, Executor, WorkflowBuilder, WorkflowContext, WorkflowEvent, handler
from typing_extensions import Never


class Classifier(Executor):
    """Classifies support tickets into categories."""

    def __init__(self, chat_client, id: str = "classifier"):
        super().__init__(id=id)
        self.agent = Agent(
            client=chat_client,
            name="classifier",
            instructions=(
                "You classify support tickets into exactly one category: 'billing', 'technical', or 'general'. "
                "Respond with ONLY the category name, nothing else."
            ),
        )

    @handler
    async def handle(self, ticket: str, ctx: WorkflowContext[dict, dict]) -> None:
        response = await self.agent.run(ticket)
        category = response.text.strip().lower()

        await ctx.yield_output({
            "step": "classification",
            "category": category,
            "ticket": ticket,
        })

        await ctx.send_message({
            "ticket": ticket,
            "category": category
        })



class Responder(Executor):
    """Generates responses based on ticket category."""

    def __init__(self, chat_client, id: str = "responder"):
        super().__init__(id=id)
        self.agent = Agent(
            client=chat_client,
            name="responder",
            instructions=(
                "You are a helpful support agent. Generate a professional response to the customer's ticket. "
                "Tailor your response based on the category provided. Be concise and helpful."
            ),
        )
        

    @handler
    async def handle(self, data: dict, ctx: WorkflowContext[Never, str]) -> None:
        prompt = f"Category: {data['category']}\nTicket: {data['ticket']}"
        response = await self.agent.run(prompt)

        await ctx.yield_output({
            "step": "response",
            "category": data["category"],
            "response": response.text,
        })


classifier = Classifier(client)
responder = Responder(client)

workflow = (
    WorkflowBuilder(start_executor=classifier)
    .add_edge(classifier, responder)
    .build()
)

async for event in workflow.run(
    "I was charged twice for my subscription last month. Please help!",
    stream=True,
):

    if event.type == "output":
        if event.data["step"] == "classification":
            print(f"\n Classification: {event.data['category']}")
        elif event.data["step"] == "response":
            print(f"\n Final Response:\n{event.data['response']}")

```
</details>


## Chapter 2 Workflow Exercises Complete! 🎉

 You've completed the following Chapter 3 exercises:

   1. Multi-Agent Workflow - connected three agents in a linear pipeline.
   2. Custom Executor Workflow with Streaming - built custom Executor classes that wrap 
  agents, control message flow between steps, and stream events in real-time.

  These patterns give you full control over how agents collaborate. In the next section, we'll
  explore pre-built orchestration patterns — Sequential, Concurrent, and Handoff that
  simplify common multi-agent coordination scenarios so you don't have to wire everything up by
  hand.

